# Gen Z Loyalty Analysis — American Airlines

This notebook addresses two core business problems:

**Problem 1:** Who is Gen Z compared to other generations?  
- Loyalty enrollment rate by generation  
- Engagement level by generation  
- Booking frequency by generation  
- Methods: EDA + K-Means Clustering  

**Problem 2:** What motivates Gen Z loyalty?  
- Which factors correlate most with engagement?  
- Which factors predict loyalty enrollment?  
- Methods: XGBoost, LightGBM, SHAP Values

---
## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score

import xgboost as xgb
import lightgbm as lgb
import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

In [ ]:
# Load train and test datasets
train_df = pd.read_csv('train.csv', index_col=0)
test_df = pd.read_csv('test.csv', index_col=0)

# Combine for full analysis
df = pd.concat([train_df, test_df], ignore_index=True)
print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print(f'Combined:    {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
print('Missing values:')
print(missing[missing > 0])

# Fill Arrival Delay missing values with median
df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median(), inplace=True)
print(f'\nMissing values after imputation: {df.isnull().sum().sum()}')

---
## 2. Feature Engineering — Generation Buckets

We assign generational labels based on the Age column using standard generational cutoffs.  
Since we have ages (not birth years), we approximate using 2024 as reference year:
- **Gen Z**: Age 12–27 (born ~1997–2012)
- **Millennials**: Age 28–43 (born ~1981–1996)
- **Gen X**: Age 44–59 (born ~1965–1980)
- **Boomers**: Age 60+ (born before 1965)

In [ ]:
def assign_generation(age):
    if age <= 27:
        return 'Gen Z'
    elif age <= 43:
        return 'Millennial'
    elif age <= 59:
        return 'Gen X'
    else:
        return 'Boomer'

df['Generation'] = df['Age'].apply(assign_generation)

print('Generation distribution:')
print(df['Generation'].value_counts())
print(f'\nTotal rows: {len(df)}')

In [ ]:
# Age distribution by generation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
gen_order = ['Gen Z', 'Millennial', 'Gen X', 'Boomer']
sns.countplot(data=df, x='Generation', order=gen_order, palette='viridis', ax=axes[0])
axes[0].set_title('Passenger Count by Generation')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Age histogram by generation
for gen in gen_order:
    subset = df[df['Generation'] == gen]
    axes[1].hist(subset['Age'], bins=20, alpha=0.5, label=gen)
axes[1].set_title('Age Distribution by Generation')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Derived Metrics

We engineer features that map to the analytics questions:
- **Loyalty enrollment**: `Customer Type` == "Loyal Customer"
- **Engagement score**: Mean of all 14 service rating columns (proxy for how engaged/satisfied a member is)
- **Satisfaction (binary)**: Target variable from the dataset

In [ ]:
# Define service rating columns
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Binary loyalty flag
df['is_loyal'] = (df['Customer Type'] == 'Loyal Customer').astype(int)

# Engagement score (mean of all service ratings)
df['engagement_score'] = df[service_cols].mean(axis=1)

# Satisfaction binary
df['is_satisfied'] = (df['satisfaction'] == 'satisfied').astype(int)

print('New columns added: is_loyal, engagement_score, is_satisfied')
df[['Generation', 'is_loyal', 'engagement_score', 'is_satisfied']].head(10)

---
# PROBLEM 1: Who is Gen Z Compared to Other Generations?
---
## 4. Method 1 — Exploratory Data Analysis (EDA)

### 4.1 Loyalty Enrollment Rate by Generation

In [ ]:
# Loyalty enrollment rate by generation
loyalty_rate = df.groupby('Generation')['is_loyal'].mean().reindex(gen_order)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(loyalty_rate.index, loyalty_rate.values, color=sns.color_palette('viridis', 4))
ax.set_title('Loyalty Enrollment Rate by Generation', fontsize=14, fontweight='bold')
ax.set_ylabel('Enrollment Rate')
ax.set_ylim(0, 1)
for bar, val in zip(bars, loyalty_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Loyalty Enrollment Rate by Generation:')
print(loyalty_rate.apply(lambda x: f'{x:.1%}'))

### 4.2 Engagement Level by Generation

In [ ]:
# Engagement score distribution by generation — boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=df, x='Generation', y='engagement_score', order=gen_order,
            palette='viridis', ax=axes[0])
axes[0].set_title('Engagement Score Distribution by Generation', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Engagement Score (Mean Service Rating)')

# Violin plot for richer view
sns.violinplot(data=df, x='Generation', y='engagement_score', order=gen_order,
               palette='viridis', inner='quartile', ax=axes[1])
axes[1].set_title('Engagement Score Spread by Generation', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Engagement Score')

plt.tight_layout()
plt.show()

# Summary statistics
engagement_stats = df.groupby('Generation')['engagement_score'].describe().reindex(gen_order)
print('Engagement Score Summary by Generation:')
engagement_stats

### 4.3 Satisfaction Rate by Generation

In [ ]:
# Satisfaction rate by generation
sat_rate = df.groupby('Generation')['is_satisfied'].mean().reindex(gen_order)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(sat_rate.index, sat_rate.values, color=sns.color_palette('magma', 4))
ax.set_title('Satisfaction Rate by Generation', fontsize=14, fontweight='bold')
ax.set_ylabel('Satisfaction Rate')
ax.set_ylim(0, 1)
for bar, val in zip(bars, sat_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4 Generation Comparison Table

In [ ]:
# Comprehensive generation comparison table
comparison = df.groupby('Generation').agg(
    count=('id', 'count'),
    loyalty_rate=('is_loyal', 'mean'),
    satisfaction_rate=('is_satisfied', 'mean'),
    avg_engagement=('engagement_score', 'mean'),
    median_engagement=('engagement_score', 'median'),
    avg_flight_distance=('Flight Distance', 'mean'),
    avg_departure_delay=('Departure Delay in Minutes', 'mean'),
    avg_arrival_delay=('Arrival Delay in Minutes', 'mean')
).reindex(gen_order)

comparison['loyalty_rate'] = comparison['loyalty_rate'].apply(lambda x: f'{x:.1%}')
comparison['satisfaction_rate'] = comparison['satisfaction_rate'].apply(lambda x: f'{x:.1%}')

print('=== Generation Comparison Table ===')
comparison

### 4.5 Service Ratings Heatmap by Generation

In [ ]:
# Average service rating by generation — heatmap
service_by_gen = df.groupby('Generation')[service_cols].mean().reindex(gen_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(service_by_gen, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Rating (0-5)'})
ax.set_title('Average Service Rating by Generation', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 4.6 Travel Class and Type Distribution by Generation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
class_dist = pd.crosstab(df['Generation'], df['Class'], normalize='index').reindex(gen_order)
class_dist.plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Travel Class Distribution by Generation', fontweight='bold')
axes[0].set_ylabel('Proportion')
axes[0].legend(title='Class', bbox_to_anchor=(1.0, 1.0))
axes[0].tick_params(axis='x', rotation=0)

# Travel type distribution
travel_dist = pd.crosstab(df['Generation'], df['Type of Travel'], normalize='index').reindex(gen_order)
travel_dist.plot(kind='bar', stacked=True, ax=axes[1], colormap='coolwarm')
axes[1].set_title('Travel Type Distribution by Generation', fontweight='bold')
axes[1].set_ylabel('Proportion')
axes[1].legend(title='Type', bbox_to_anchor=(1.0, 1.0))
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 4.7 Statistical Tests — Are Generational Differences Significant?

In [ ]:
# Chi-square test: Is loyalty enrollment independent of generation?
contingency = pd.crosstab(df['Generation'], df['is_loyal'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-Square Test — Loyalty Enrollment vs Generation')
print(f'  Chi2 = {chi2:.2f}, p-value = {p_val:.2e}, df = {dof}')
print(f'  {"Significant" if p_val < 0.05 else "Not significant"} at alpha=0.05\n')

# ANOVA: Is mean engagement score different across generations?
groups = [group['engagement_score'].values for _, group in df.groupby('Generation')]
f_stat, p_val_anova = stats.f_oneway(*groups)
print(f'One-Way ANOVA — Engagement Score across Generations')
print(f'  F = {f_stat:.2f}, p-value = {p_val_anova:.2e}')
print(f'  {"Significant" if p_val_anova < 0.05 else "Not significant"} at alpha=0.05\n')

# Chi-square test: satisfaction vs generation
contingency_sat = pd.crosstab(df['Generation'], df['is_satisfied'])
chi2_sat, p_val_sat, dof_sat, _ = stats.chi2_contingency(contingency_sat)
print(f'Chi-Square Test — Satisfaction vs Generation')
print(f'  Chi2 = {chi2_sat:.2f}, p-value = {p_val_sat:.2e}, df = {dof_sat}')
print(f'  {"Significant" if p_val_sat < 0.05 else "Not significant"} at alpha=0.05')

### 4.8 Correlation Heatmap — Service Features

In [ ]:
# Correlation heatmap of service features + key outcomes
corr_cols = service_cols + ['Flight Distance', 'Departure Delay in Minutes',
                            'Arrival Delay in Minutes', 'is_loyal', 'is_satisfied',
                            'engagement_score']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap — Service Features and Outcomes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Method 2 — K-Means Clustering

K-Means finds behavioral segments that cut across generational lines. We then examine which generation dominates each cluster.

### 5.1 Prepare Clustering Features

In [ ]:
# Clustering features — behavioral signals
cluster_features = service_cols + ['Flight Distance', 'Departure Delay in Minutes',
                                   'Arrival Delay in Minutes', 'Age']

X_cluster = df[cluster_features].copy()

# Standardize — critical for K-Means (distance-based)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f'Clustering input shape: {X_scaled.shape}')
print(f'Features: {cluster_features}')

### 5.2 Elbow Chart and Silhouette Analysis

In [ ]:
# Elbow chart + silhouette score
K_range = range(2, 9)
inertias = []
silhouette_scores = []

# Use a sample for speed
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=min(20000, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_sample)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_sample, km.labels_, sample_size=5000))
    print(f'K={k}: inertia={km.inertia_:.0f}, silhouette={silhouette_scores[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_title('Elbow Chart', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')

axes[1].plot(K_range, silhouette_scores, 'rs-', linewidth=2)
axes[1].set_title('Silhouette Score by K', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

### 5.3 Fit Final K-Means Model (K=4)

In [ ]:
# Final model with K=4 (adjust based on elbow/silhouette results above)
OPTIMAL_K = 4
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10, max_iter=300)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

### 5.4 Cluster Profiles — Behavioral Personas

In [ ]:
# Cluster profiles
profile_cols = service_cols + ['Flight Distance', 'Age', 'is_loyal', 'is_satisfied', 'engagement_score']
cluster_profiles = df.groupby('cluster')[profile_cols].mean()

print('=== Cluster Behavioral Profiles ===')
cluster_profiles.T

In [ ]:
# Heatmap of cluster centers (standardized)
centers_df = pd.DataFrame(kmeans_final.cluster_centers_, columns=cluster_features)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(centers_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('K-Means Cluster Centers (Standardized Features)', fontsize=14, fontweight='bold')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

### 5.5 Generation Composition of Each Cluster

In [ ]:
# Generation composition within each cluster
gen_cluster = pd.crosstab(df['cluster'], df['Generation'], normalize='index')[gen_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gen_cluster.plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Generation Composition by Cluster', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Proportion')
axes[0].set_xlabel('Cluster')
axes[0].legend(title='Generation')
axes[0].tick_params(axis='x', rotation=0)

# Cluster distribution within each generation
cluster_gen = pd.crosstab(df['Generation'], df['cluster'], normalize='index').reindex(gen_order)
cluster_gen.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_title('Cluster Distribution within Each Generation', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Proportion')
axes[1].set_xlabel('Generation')
axes[1].legend(title='Cluster')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\nGeneration composition of each cluster:')
gen_cluster.applymap(lambda x: f'{x:.1%}')

In [ ]:
# Assign descriptive persona labels based on cluster profiles
# (Review cluster_profiles above and assign meaningful names)
cluster_loyalty = df.groupby('cluster')['is_loyal'].mean()
cluster_engagement = df.groupby('cluster')['engagement_score'].mean()
cluster_satisfaction = df.groupby('cluster')['is_satisfied'].mean()

persona_summary = pd.DataFrame({
    'Loyalty Rate': cluster_loyalty.apply(lambda x: f'{x:.1%}'),
    'Avg Engagement': cluster_engagement.round(2),
    'Satisfaction Rate': cluster_satisfaction.apply(lambda x: f'{x:.1%}'),
    'Size': df['cluster'].value_counts().sort_index()
})

print('=== Cluster Persona Summary ===')
persona_summary

---
# PROBLEM 2: What Motivates Gen Z Loyalty?
---
## 6. Data Preparation for Predictive Modeling

Target variable: `is_loyal` (loyalty enrollment)  
Features: 14 service ratings + Flight Distance + Delays + Class + Travel Type  
We filter to Gen Z for the primary models, then compare against other generations.

In [ ]:
# Prepare modeling dataset
# Encode categorical features
df['class_encoded'] = LabelEncoder().fit_transform(df['Class'])
df['travel_type_encoded'] = LabelEncoder().fit_transform(df['Type of Travel'])
df['gender_encoded'] = LabelEncoder().fit_transform(df['Gender'])

# Feature set for modeling
model_features = service_cols + [
    'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
    'Age', 'class_encoded', 'travel_type_encoded', 'gender_encoded'
]

print(f'Model features ({len(model_features)}): {model_features}')

In [ ]:
# Gen Z subset
df_genz = df[df['Generation'] == 'Gen Z'].copy()
print(f'Gen Z sample size: {len(df_genz)}')
print(f'Gen Z loyalty rate: {df_genz["is_loyal"].mean():.1%}')

# Full dataset for comparison
X_full = df[model_features]
y_full = df['is_loyal']

# Gen Z split
X_genz = df_genz[model_features]
y_genz = df_genz['is_loyal']

X_train_gz, X_test_gz, y_train_gz, y_test_gz = train_test_split(
    X_genz, y_genz, test_size=0.2, random_state=42, stratify=y_genz
)

# Full data split for comparison
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print(f'\nGen Z train/test: {X_train_gz.shape[0]} / {X_test_gz.shape[0]}')
print(f'Full train/test:  {X_train_all.shape[0]} / {X_test_all.shape[0]}')

---
## 7. Model A — XGBoost

In [ ]:
# XGBoost — Gen Z Model
xgb_model_gz = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,      # L1 regularization
    reg_lambda=1.0,     # L2 regularization
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=20
)

xgb_model_gz.fit(
    X_train_gz, y_train_gz,
    eval_set=[(X_test_gz, y_test_gz)],
    verbose=False
)

# Predictions
y_pred_xgb_gz = xgb_model_gz.predict(X_test_gz)
y_proba_xgb_gz = xgb_model_gz.predict_proba(X_test_gz)[:, 1]

print('=== XGBoost — Gen Z Results ===')
print(f'ROC-AUC: {roc_auc_score(y_test_gz, y_proba_xgb_gz):.4f}')
print(f'\n{classification_report(y_test_gz, y_pred_xgb_gz, target_names=["Not Loyal", "Loyal"])}')

In [ ]:
# Cross-validation score for robustness
xgb_cv_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, eval_metric='logloss'
)
cv_scores_xgb = cross_val_score(xgb_cv_model, X_genz, y_genz, cv=5, scoring='roc_auc')
print(f'XGBoost 5-Fold CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std():.4f})')

In [ ]:
# XGBoost Feature Importance (Gain)
xgb_importance = pd.Series(
    xgb_model_gz.feature_importances_,
    index=model_features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
xgb_importance.plot(kind='barh', color=sns.color_palette('viridis', len(xgb_importance)), ax=ax)
ax.set_title('XGBoost Feature Importance — Gen Z Loyalty Drivers', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance (Gain)')
plt.tight_layout()
plt.show()

print('\nTop 5 XGBoost Loyalty Drivers for Gen Z:')
print(xgb_importance.sort_values(ascending=False).head(5))

---
## 8. Model B — LightGBM

In [ ]:
# LightGBM — Gen Z Model
lgb_model_gz = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

lgb_model_gz.fit(
    X_train_gz, y_train_gz,
    eval_set=[(X_test_gz, y_test_gz)],
    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)]
)

# Predictions
y_pred_lgb_gz = lgb_model_gz.predict(X_test_gz)
y_proba_lgb_gz = lgb_model_gz.predict_proba(X_test_gz)[:, 1]

print('=== LightGBM — Gen Z Results ===')
print(f'ROC-AUC: {roc_auc_score(y_test_gz, y_proba_lgb_gz):.4f}')
print(f'\n{classification_report(y_test_gz, y_pred_lgb_gz, target_names=["Not Loyal", "Loyal"])}')

In [ ]:
# LightGBM Cross-validation
lgb_cv_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbose=-1
)
cv_scores_lgb = cross_val_score(lgb_cv_model, X_genz, y_genz, cv=5, scoring='roc_auc')
print(f'LightGBM 5-Fold CV ROC-AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})')

In [ ]:
# LightGBM Feature Importance
lgb_importance = pd.Series(
    lgb_model_gz.feature_importances_,
    index=model_features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
lgb_importance.plot(kind='barh', color=sns.color_palette('magma', len(lgb_importance)), ax=ax)
ax.set_title('LightGBM Feature Importance — Gen Z Loyalty Drivers', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance (Split Count)')
plt.tight_layout()
plt.show()

print('\nTop 5 LightGBM Loyalty Drivers for Gen Z:')
print(lgb_importance.sort_values(ascending=False).head(5))

### 8.1 Model Agreement — XGBoost vs LightGBM

In [ ]:
# Compare feature importance rankings between the two models
xgb_rank = xgb_importance.sort_values(ascending=False).reset_index()
xgb_rank.columns = ['Feature', 'XGBoost_Importance']
xgb_rank['XGBoost_Rank'] = range(1, len(xgb_rank) + 1)

lgb_rank = lgb_importance.sort_values(ascending=False).reset_index()
lgb_rank.columns = ['Feature', 'LightGBM_Importance']
lgb_rank['LightGBM_Rank'] = range(1, len(lgb_rank) + 1)

comparison_df = xgb_rank.merge(lgb_rank, on='Feature')
comparison_df = comparison_df.sort_values('XGBoost_Rank')

print('=== Feature Importance Ranking Comparison ===')
print(comparison_df[['Feature', 'XGBoost_Rank', 'LightGBM_Rank']].to_string(index=False))

# Rank correlation
rank_corr, rank_p = stats.spearmanr(comparison_df['XGBoost_Rank'], comparison_df['LightGBM_Rank'])
print(f'\nSpearman Rank Correlation: {rank_corr:.4f} (p={rank_p:.4e})')
print(f'Model Agreement: {"Strong" if rank_corr > 0.7 else "Moderate" if rank_corr > 0.4 else "Weak"}')

In [ ]:
# Side-by-side feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

xgb_importance.sort_values(ascending=True).plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('XGBoost — Gen Z', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance (Gain)')

lgb_importance.sort_values(ascending=True).plot(
    kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('LightGBM — Gen Z', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance (Split Count)')

plt.suptitle('Feature Importance Comparison — Gen Z Loyalty Drivers',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. SHAP Values — Explaining Individual Predictions

SHAP moves from "what matters globally" to "how much each feature contributed to THIS member's prediction."

In [ ]:
# SHAP for XGBoost
explainer_xgb = shap.TreeExplainer(xgb_model_gz)
shap_values_xgb = explainer_xgb.shap_values(X_test_gz)

print('SHAP values computed for XGBoost Gen Z model.')
print(f'Shape: {shap_values_xgb.shape}')

In [ ]:
# SHAP Summary Plot — XGBoost (Global feature importance + direction)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_xgb, X_test_gz, feature_names=model_features, show=False)
plt.title('SHAP Summary — XGBoost Gen Z Loyalty Drivers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot — Mean absolute SHAP values
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_xgb, X_test_gz, feature_names=model_features,
                  plot_type='bar', show=False)
plt.title('Mean |SHAP| — XGBoost Gen Z Feature Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP for LightGBM
explainer_lgb = shap.TreeExplainer(lgb_model_gz)
shap_values_lgb = explainer_lgb.shap_values(X_test_gz)

# LightGBM binary classifier returns list of 2 arrays — take class 1 (loyal)
if isinstance(shap_values_lgb, list):
    shap_values_lgb = shap_values_lgb[1]

print('SHAP values computed for LightGBM Gen Z model.')

In [ ]:
# SHAP Summary Plot — LightGBM
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_lgb, X_test_gz, feature_names=model_features, show=False)
plt.title('SHAP Summary — LightGBM Gen Z Loyalty Drivers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.1 SHAP Dependence Plots — Key Loyalty Drivers

In [ ]:
# SHAP dependence plots for top features
# Identify top 4 features by mean absolute SHAP
mean_abs_shap = np.abs(shap_values_xgb).mean(axis=0)
top_features_idx = np.argsort(mean_abs_shap)[::-1][:4]
top_feature_names = [model_features[i] for i in top_features_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, (feat_name, ax) in enumerate(zip(top_feature_names, axes.flatten())):
    shap.dependence_plot(
        feat_name, shap_values_xgb, X_test_gz,
        feature_names=model_features, ax=ax, show=False
    )
    ax.set_title(f'SHAP Dependence: {feat_name}', fontweight='bold')

plt.suptitle('SHAP Dependence Plots — Top 4 Gen Z Loyalty Drivers',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 9.2 SHAP Individual Prediction Explanation

In [ ]:
# Explain a specific Gen Z member's prediction
# Pick a member predicted as loyal and one predicted as not loyal
loyal_idx = np.where(y_pred_xgb_gz == 1)[0][0]
not_loyal_idx = np.where(y_pred_xgb_gz == 0)[0][0]

print(f'Explaining prediction for Gen Z member predicted as LOYAL (index {loyal_idx}):')
print(f'  Predicted probability: {y_proba_xgb_gz[loyal_idx]:.3f}\n')

# Waterfall plot for loyal member
shap.initjs()
explanation_xgb = shap.Explanation(
    values=shap_values_xgb[loyal_idx],
    base_values=explainer_xgb.expected_value,
    data=X_test_gz.iloc[loyal_idx].values,
    feature_names=model_features
)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(explanation_xgb, show=False)
plt.title('SHAP Waterfall — Gen Z Loyal Member', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot for non-loyal member
print(f'Explaining prediction for Gen Z member predicted as NOT LOYAL (index {not_loyal_idx}):')
print(f'  Predicted probability: {y_proba_xgb_gz[not_loyal_idx]:.3f}\n')

explanation_not_loyal = shap.Explanation(
    values=shap_values_xgb[not_loyal_idx],
    base_values=explainer_xgb.expected_value,
    data=X_test_gz.iloc[not_loyal_idx].values,
    feature_names=model_features
)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(explanation_not_loyal, show=False)
plt.title('SHAP Waterfall — Gen Z Non-Loyal Member', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Model Performance Summary

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_xgb_gz, y_pred_lgb_gz],
                              ['XGBoost', 'LightGBM']):
    cm = confusion_matrix(y_test_gz, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Loyal', 'Loyal'],
                yticklabels=['Not Loyal', 'Loyal'])
    ax.set_title(f'{title} — Confusion Matrix (Gen Z)', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Final model comparison table
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame({
    'Model': ['XGBoost (Gen Z)', 'LightGBM (Gen Z)'],
    'ROC-AUC': [
        roc_auc_score(y_test_gz, y_proba_xgb_gz),
        roc_auc_score(y_test_gz, y_proba_lgb_gz)
    ],
    'Accuracy': [
        accuracy_score(y_test_gz, y_pred_xgb_gz),
        accuracy_score(y_test_gz, y_pred_lgb_gz)
    ],
    'F1-Score': [
        f1_score(y_test_gz, y_pred_xgb_gz),
        f1_score(y_test_gz, y_pred_lgb_gz)
    ],
    'CV ROC-AUC (5-Fold)': [
        f'{cv_scores_xgb.mean():.4f} +/- {cv_scores_xgb.std():.4f}',
        f'{cv_scores_lgb.mean():.4f} +/- {cv_scores_lgb.std():.4f}'
    ]
})

print('=== Final Model Performance Comparison ===')
results

---
## 11. Key Findings and Business Recommendations

### Problem 1 Findings — Who is Gen Z?
- **Loyalty enrollment rate**: Compare Gen Z vs Millennials vs Gen X from the bar chart above
- **Engagement level**: Review the boxplots — is Gen Z's engagement score spread wider or lower than other cohorts?
- **Service preferences**: The heatmap reveals which services Gen Z rates higher/lower
- **Behavioral clusters**: K-Means identifies segments that cut across generations — does Gen Z concentrate in specific clusters?

### Problem 2 Findings — What Motivates Gen Z Loyalty?
- **Top loyalty drivers**: Both XGBoost and LightGBM rank features — check for agreement in the top 3-5
- **SHAP insights**: The summary plot shows both magnitude AND direction — which features push Gen Z toward loyalty vs away?
- **Individual explanations**: Waterfall plots show how specific member profiles drive predictions

### Actionable Recommendations
1. Review the top SHAP-identified loyalty drivers and invest in those service areas
2. Use cluster personas to design targeted campaigns for Gen Z segments
3. Where models disagree on feature importance, investigate segment-specific effects
4. Use the enrollment probability scores to identify high-potential Gen Z non-members for outreach

In [ ]:
# Final summary of top Gen Z loyalty drivers (agreed by both models)
print('=== TOP GEN Z LOYALTY DRIVERS ===')
print('Features ranked by both XGBoost and LightGBM:\n')

for _, row in comparison_df.head(10).iterrows():
    agreement = 'AGREED' if abs(row['XGBoost_Rank'] - row['LightGBM_Rank']) <= 2 else 'DIVERGENT'
    print(f"  {row['Feature']:40s}  XGB Rank: {int(row['XGBoost_Rank']):2d}  "
          f"LGB Rank: {int(row['LightGBM_Rank']):2d}  [{agreement}]")